<a href="https://colab.research.google.com/github/hosseinta2/LLM-from-scratch/blob/main/LLM_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
print(os.listdir())

['.config', 'story.txt', 'sample_data']


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Fri May 22 17:59:27 2026

@author: hosseintaheri
"""
import re
with open("story.txt", "r",encoding = "utf-8") as f:
    file = f.read()
print("number of chars:", len(file))

sample = file[:100]
print(sample)

result = re.split(r'([,.?_!"()\']|--|\s)', file) # tokenzed input
result = [x for x in result if x!=' ']
print(len(result))
print(result[:30])

all_words = sorted(list(set(result)))
vocab_size = len(all_words)
print(vocab_size)


vocab = {word:num for num,word in enumerate(all_words) } # for encoding

ids = [vocab[x] for x in result] # index for each token
print(ids[:20])

## final code

class tokenizer:
    def __init__(self, vocab):
        self.encoder = vocab
        self.decoder ={num:word for word,num in vocab.items()}
    def encode(self, text):
        tokenized = re.split(r'([,.?_!"()\']|--|\s)', text)
        tokenized = [x for x in tokenized if x!= ' ']
        ids =  [self.encoder[x] for x in tokenized]
        return ids
    def decode(self, ids):
        text = " ".join(self.decoder[x] for x in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

tkz = tokenizer(vocab)
result1 = tkz.encode(file)
print(result1[:10])
result2 = tkz.decode(result1)
print(result2[:1000])


number of chars: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g
5598
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', '']
1161
[57, 48, 156, 1030, 61, 41, 841, 121, 265, 496, 8, 1029, 121, 510, 445, 401, 8, 935, 598, 1106]
[57, 48, 156, 1030, 61, 41, 841, 121, 265, 496]
I HAD always thought Jack Gisburn rather a cheap genius -- though a good fellow enough -- so it was no great surprise to me to hear that,  in the height of his glory,  he had dropped his painting,  married a rich widow,  and established himself in a villa on the Riviera.( Though I rather thought it would have been Rome or Florence.)" The height of his glory"  -- that was what the women called it.  I can hear Mrs.  Gideon Thwing -- his last Chicago sitter -- deploring his unaccountable a

adding uknown tokens and end of text tokens to the vocab list and updating encoder

In [5]:
all_words = all_words + ["<|endoftext|>", "<|unk|>"]
vocab = {word:num for num,word in enumerate(all_words)}
class tokenizer:
    def __init__(self, vocab):
        self.encoder = vocab
        self.decoder ={num:word for word,num in vocab.items()}
    def encode(self, text):
        tokenized = re.split(r'([,.?_!"()\']|--|\s)', text)
        tokenized = [x for x in tokenized if x!= ' ']
        tokenized = [x if x in self.encoder else '<|unk|>' for x in tokenized ]
        ids =  [self.encoder[x] for x in tokenized]
        return ids
    def decode(self, ids):
        text = " ".join(self.decoder[x] for x in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
tkz = tokenizer(vocab)
print(tkz.decode(tkz.encode("Hello, my beloved friend, I had always thought of you <|endoftext|>")))


<|unk|>,  my <|unk|> friend,  I had always thought of you <|endoftext|>


Implementing Byte pair token

In [6]:
pip install tiktoken

In [7]:
from importlib.metadata import version

import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [8]:
tokenizer = tiktoken.get_encoding("gpt2")

In [9]:
ids = tokenizer.encode(file)
print(ids[:10])

[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]


In [10]:
print(tokenizer.decode(ids)[:10])
print(len(ids))

I HAD alwa
5145


In [11]:
context_size = 10
for i in range(1,context_size):
  x = ids[:i]
  y = [ids[i]]
  print(tokenizer.decode(x),tokenizer.decode(y))

I  H
I H AD
I HAD  always
I HAD always  thought
I HAD always thought  Jack
I HAD always thought Jack  G
I HAD always thought Jack G is
I HAD always thought Jack Gis burn
I HAD always thought Jack Gisburn  rather


In [12]:

import torch
from torch.utils.data import Dataset, DataLoader

class GPTdataset(Dataset):
  def __init__(self,txt,tokenizer,context_size,stride):
      self.input_ids = []
      self.target_ids = []

      ids = tokenizer.encode(txt)
      for i in range(0,len(ids)-context_size,stride):
        x = ids[i:i+context_size]
        y = ids[i+1:i+context_size+1]
        self.input_ids.append(torch.tensor(x))
        self.target_ids.append(torch.tensor(y))
  def __len__(self): #C
      return len(self.input_ids)

  def __getitem__(self, idx): #D
        return self.input_ids[idx], self.target_ids[idx]


In [13]:
data = GPTdataset(file,tokenizer,32,1)

In [14]:
def dataloder_v1(txt,tokenizer,context_size,stride,batch_size,shuffle,drop_last):
  dataset = GPTdataset(file,tokenizer=tokenizer,context_size=context_size,stride=stride)
  data = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last)
  return data

In [16]:
tokenizer=tiktoken.get_encoding("gpt2")

data = dataloder_v1(file,tokenizer=tokenizer, context_size=4,stride=1,batch_size=2,shuffle=False,drop_last=True)

In [17]:
i=0
for batch in data:
  print(batch)
  i+=1
  if i==2:
     break

data_iter = iter(data)
input,output = next(data_iter)
print(input,output)

[tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807]]), tensor([[ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619]])]
[tensor([[2885, 1464, 1807, 3619],
        [1464, 1807, 3619,  402]]), tensor([[1464, 1807, 3619,  402],
        [1807, 3619,  402,  271]])]
tensor([[  40,  367, 2885, 1464],
        [ 367, 2885, 1464, 1807]]) tensor([[ 367, 2885, 1464, 1807],
        [2885, 1464, 1807, 3619]])


Creating the Embedding layer

In [18]:
torch.manual_seed(123)
embedding_layer=torch.nn.Embedding(num_embeddings=50257,embedding_dim=256)


context_size =4
batch_size = 8
data = dataloder_v1(file,tokenizer,context_size=context_size,stride=context_size,batch_size=batch_size,shuffle=False,drop_last=False)
data_iter=iter(data)
input, output = next(data_iter)
print("Token IDs:\n", input)
print("\nInputs shape:\n", input.shape)

embedded_input = embedding_layer(input)
print("\nInputs shape:\n", embedded_input.shape)


Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])

Inputs shape:
 torch.Size([8, 4, 256])


pos embedding:

In [20]:
context_size = 4
positional_layer = torch.nn.Embedding(num_embeddings=context_size,embedding_dim=256)
pos_embeddings = positional_layer(torch.arange(context_size))
print(pos_embeddings.shape)
final_embedding = pos_embeddings+ embedded_input
print(final_embedding.shape)

torch.Size([4, 256])
torch.Size([8, 4, 256])


In [25]:
import torch
x = torch.tensor([[1,2,3],[3,2,1]])
torch.exp(x).sum(dim=0)
torch.exp(x)/torch.exp(x).sum(dim=0)


tensor([[0.1192, 0.5000, 0.8808],
        [0.8808, 0.5000, 0.1192]])

In [22]:
import torch
x = torch.tensor([[1,2,3],[3,2,1]])
torch.exp(x).sum(dim=1).unsqueeze(dim=0)
torch.exp(x)/torch.transpose(torch.exp(x).sum(dim=1).unsqueeze(dim=0),0,1)

tensor([[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]])

In [23]:
torch.softmax(x.float(),dim=0)

tensor([[0.1192, 0.5000, 0.8808],
        [0.8808, 0.5000, 0.1192]])

In [24]:
torch.softmax(x.float(),dim=1)

tensor([[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]])

In [26]:
inputs = torch.tensor([[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]])
d_in = inputs.shape[1]

In [27]:
d_out = 10

In [ ]:
d_in

3

In [28]:
W_query = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)

In [29]:
x_q,x_k,x_v = inputs@W_query,inputs@W_key,inputs@W_value


Final Attention code:

In [30]:
import torch.nn as nn
class Attention(nn.Module):
  def __init__(self,d_in,d_out):
     super().__init__()
     self.W_key   = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=True)
     self.W_query = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=True)
     self.W_value = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=True)
  def forward(self,inputs):
     X_query= inputs@self.W_query
     X_value= inputs@self.W_value
     X_key= inputs@self.W_key
     d_out = X_key.shape[-1]
     atten_scores = X_query@X_value.T
     atten_weights= torch.softmax(atten_scores/d_out**0.5, dim=1)
     output = atten_weights@X_value
     return output


inputs = torch.tensor([[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]])
att = Attention(d_in=inputs.shape[1],d_out=10)
out = att(inputs)
out.shape

torch.Size([2, 10])

In [31]:
import torch.nn as nn
class Attention_v2(nn.Module):
  def __init__(self,d_in,d_out,b):
     super().__init__()
     self.W_key   = torch.nn.Linear(d_in,d_out,bias=b)
     self.W_query = torch.nn.Linear(d_in,d_out,bias=b)
     self.W_value = torch.nn.Linear(d_in,d_out,bias=b)
  def forward(self,inputs):
     X_query= self.W_query(inputs)
     X_value= self.W_value(inputs)
     X_key= self.W_key(inputs)
     d_out = X_key.shape[-1]
     atten_scores = X_query@X_value.T
     atten_weights= torch.softmax(atten_scores/d_out**0.5, dim=1)
     output = atten_weights@X_value
     return output


inputs = torch.tensor([[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]])
att = Attention_v2(d_in=inputs.shape[1],d_out=10,b=False)
out = att(inputs)
out.shape

torch.Size([2, 10])

Causal attention, with batch and dropout:

In [32]:
x = torch.tensor([[1,2],[3,4]])
mask = torch.triu(torch.ones(2,2))
masked_input = x.float().masked_fill(mask.bool(), -torch.inf)
masked_input

tensor([[-inf, -inf],
        [3., -inf]])

In [33]:
import torch.nn as nn
import torch
class Attention_v2(nn.Module):
  def __init__(self,d_in,d_out,bias,dropout_rate,context_size):
     super().__init__()
     self.W_key   = torch.nn.Linear(d_in,d_out,bias=bias)
     self.W_query = torch.nn.Linear(d_in,d_out,bias=bias)
     self.W_value = torch.nn.Linear(d_in,d_out,bias=bias)
     self.drop = nn.Dropout(dropout_rate)
     self.register_buffer('mask',torch.triu(torch.ones(context_size,context_size),diagonal=1))

  def forward(self,inputs):
     X_query= self.W_query(inputs)
     X_value= self.W_value(inputs)
     X_key= self.W_key(inputs)
     d_out = X_key.shape[-1]
     atten_scores = X_query@torch.transpose(X_key,1,2)
     context_size = atten_scores.shape[-1]
     atten_masked = atten_scores.masked_fill(self.mask.bool()[:context_size,:context_size],-torch.inf)
     atten_weights= torch.softmax(atten_masked/d_out**0.5, dim=-1)
     atten_weights_drop = self.drop(atten_weights)
     output = atten_weights_drop@X_value
     return output


inputs = torch.tensor([[[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]],[[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]],[[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]]])
att = Attention_v2(d_in=inputs.shape[-1],d_out=10,bias=False,dropout_rate=0.2,context_size=inputs.shape[1])
out = att(inputs)
out.shape

torch.Size([3, 2, 10])

Multi-head attention:

In [35]:
class multi_head_attention(nn.Module):
  def __init__(self, d_in,d_out,bias,dropout_rate,context_size,heads):
     super().__init__()
     self.heads = nn.ModuleList([Attention_v2(d_in,d_out,bias,dropout_rate,context_size) for _ in range(heads)])
  def forward(self,inputs):
    return torch.cat([head(inputs) for head in self.heads],dim=-1)

inputs = torch.tensor([[[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]],[[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]],[[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]]])


num_heads =2
att = multi_head_attention(d_in=inputs.shape[-1],d_out=10,bias=False,dropout_rate=0.2,context_size=inputs.shape[1],heads=num_heads)
out = att(inputs)
out.shape

torch.Size([3, 2, 20])

Another implementation of MLA

In [54]:
class MHA_v2(nn.Module):
  def __init__(self,d_in,d_out,b,do_rate,num_heads,context_size):
     super().__init__()
     assert d_out%num_heads==0
     self.head_dim = d_out//num_heads
     self.num_heads = num_heads
     self.W_k = nn.Linear(d_in,d_out,bias=b)
     self.W_q = nn.Linear(d_in,d_out,bias=b)
     self.W_v = nn.Linear(d_in,d_out,bias=b)
     self.drop_out= nn.Dropout(do_rate)
     self.register_buffer('mask',torch.triu(torch.ones(context_size,context_size),1))
     self.projection = nn.Linear(d_out,d_out)
  def forward(self,x):
    X_k = self.W_k(x)
    X_q = self.W_q(x)
    X_v = self.W_v(x)
    batch_size,context_size,d_out = X_k.shape
    head_dim = self.head_dim
    X_k = X_k.view(batch_size, context_size, self.num_heads, head_dim)
    X_q = X_q.view(batch_size, context_size, self.num_heads, head_dim)
    X_v = X_v.view(batch_size, context_size, self.num_heads, head_dim)
    X_k = X_k.transpose(1,2)
    X_q = X_q.transpose(1,2)
    X_v = X_v.transpose(1,2)
    atten_prod = X_q@X_k.transpose(2,3)
    atten_prod.masked_fill(self.mask.bool()[:context_size,:context_size],-torch.inf)
    atten_scores = torch.softmax(atten_prod/X_k.shape[-1]**0.5,dim=-1)
    atten_scores = self.drop_out(atten_scores)
    output = atten_scores@X_v # batch*num_heads*context_size*head_head_dim
    output = output.transpose(1,2) # batch*context_size*num_heads*head_head_dim
    d_out= self.num_heads*head_dim
    output = output.contiguous().view(batch_size,context_size,d_out)
    output = self.projection(output)
    return output
inputs = torch.tensor([[[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]],[[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]],[[0.0900, 0.2447, 0.6652],
        [0.6652, 0.2447, 0.0900]]])
inputs.shape

model = MHA_v2(d_in=inputs.shape[-1], d_out= 10, b=False, do_rate=0,num_heads=2,context_size=inputs.shape[1])
output = model(inputs)
print(output.shape)
output



torch.Size([3, 2, 10])


tensor([[[ 0.2606, -0.3684,  0.0366,  0.1461, -0.2289,  0.2176,  0.2020,
           0.1316,  0.1519,  0.4404],
         [ 0.2587, -0.3641,  0.0354,  0.1455, -0.2303,  0.2167,  0.2012,
           0.1339,  0.1511,  0.4412]],

        [[ 0.2606, -0.3684,  0.0366,  0.1461, -0.2289,  0.2176,  0.2020,
           0.1316,  0.1519,  0.4404],
         [ 0.2587, -0.3641,  0.0354,  0.1455, -0.2303,  0.2167,  0.2012,
           0.1339,  0.1511,  0.4412]],

        [[ 0.2606, -0.3684,  0.0366,  0.1461, -0.2289,  0.2176,  0.2020,
           0.1316,  0.1519,  0.4404],
         [ 0.2587, -0.3641,  0.0354,  0.1455, -0.2303,  0.2167,  0.2012,
           0.1339,  0.1511,  0.4412]]], grad_fn=<ViewBackward0>)